# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore a complex Croissant dataset using the `mlcroissant` library, including listing available record sets and fields, loading data into DataFrames, and performing typical preprocessing and exploratory analysis.

### Dataset Source
The dataset is loaded directly from its Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset and its metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Dataset object and display metadata
dataset = mlc.Dataset(croissant_url)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review the record sets and their fields/columns by `@id`.

In [ ]:
# Print all record sets and their fields with @id

for record_set in dataset.metadata.record_sets:
    print(f"--- Record Set: {record_set.name} (@id: {record_set.id}) ---")
    for field in record_set.fields:
        col_id = field.column.id if hasattr(field, "column") and field.column is not None else None
        print(f"  - Field: {field.name} (@id: {field.id}) | Column: {col_id}")
    print("")

## 3. Data Extraction
Load the data for one or more record sets into pandas DataFrames using their `@id` references. See previous output for the available record set IDs.

In [ ]:
# List all record set @id's for extraction
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for '{record_set_id}' with columns:")
    print(df.columns.tolist())

# Select the main protein table (rename this if your primary table has a different @id; adapt below as needed)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nHead of the record set '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply filtering, normalization, and grouping on the main DataFrame. Adjust field IDs as appropriate for your data. Here we search for a numeric field (e.g., MW, peptide_count) to demonstrate the process.

In [ ]:
main_df = dataframes[main_record_set_id]

# Attempt to select a numeric field for demonstration
numeric_field_candidates = [col for col in main_df.columns if main_df[col].dtype in ['float64', 'int64', 'float32', 'int32']]
print('Numeric field candidates:', numeric_field_candidates)
# If a field is not already numeric, try to coerce a likely field
if not numeric_field_candidates:
    # Try common protein table fields, e.g., 'cr:MW', 'cr:peptide_count', 'cr:coverage' by their @id
    for field in ['cr:MW', 'cr:peptide_count', 'cr:coverage']:
        if field in main_df.columns:
            main_df[field] = pd.to_numeric(main_df[field], errors='coerce')
            numeric_field_candidates.append(field)
    print('Coerced numeric fields:', numeric_field_candidates)

# Choose a numeric field (edit this based on your actual @id):
numeric_field_id = numeric_field_candidates[0] if numeric_field_candidates else main_df.columns[0]

threshold = main_df[numeric_field_id].mean()
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean value):")
print(filtered_df[[numeric_field_id]].head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized values for {numeric_field_id}:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt grouping by another field, e.g., a categorical field (such as 'cr:description', 'cr:accession' by @id)
group_field_candidates = [col for col in main_df.columns if main_df[col].dtype == 'object' and col != numeric_field_id]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"\nMean {numeric_field_id} grouped by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the main numeric field (e.g., molecular weight or peptide count), and show a potential relationship between two fields, if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.show()

# If a second numeric field exists, plot a scatterplot
if len(numeric_field_candidates) > 1:
    plt.figure(figsize=(7,5))
    sns.scatterplot(x=main_df[numeric_field_candidates[0]], y=main_df[numeric_field_candidates[1]])
    plt.title(f'{numeric_field_candidates[0]} vs. {numeric_field_candidates[1]}')
    plt.xlabel(numeric_field_candidates[0])
    plt.ylabel(numeric_field_candidates[1])
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated the process of loading a Croissant-described dataset using `mlcroissant`, listing its record sets and fields by `@id`, extracting tables, and performing common data processing and visualization. This approach is generalizable to any Croissant dataset with well-defined record sets and fields.

**Tips:**
- Always inspect the actual record set and field `@id` references for robust downstream processing.
- For more advanced workflow, you can build automated pipelines using those `@id` references for processing, mapping, and harmonization tasks.